# `UsageMetadataCallbackHandler`

Callback handler that collects and aggregates token-usage metadata from completed chat-model calls.

Usage is stored separately for each value found in:

```python
AIMessage.response_metadata["model_name"]
```

When several calls use the same model name, their `UsageMetadata` values are combined using `add_usage`.

- Bases: `BaseCallbackHandler`
- Added in: `langchain-core` `0.3.49`

## Constructor

```python
UsageMetadataCallbackHandler()
```

The constructor does not accept any arguments.

## Attributes

* `_lock` — A `threading.Lock` used to protect updates to the shared usage dictionary.
* `usage_metadata` — Aggregated usage grouped by model name.
  * Type:
    ```python
    dict[str, UsageMetadata]
    ```
  * Initial value:
    ```python
    {}
    ```

## Stored Data

The dictionary uses this general structure:

```python
{
    "<model-name>": {
        "input_tokens": ...,
        "output_tokens": ...,
        "total_tokens": ...,
        # Optional input/output token-detail dictionaries
    }
}
```

Example:

```python
{
    "gpt-5.5": {
        "input_tokens": 125,
        "output_tokens": 48,
        "total_tokens": 173,
    },
    "claude-haiku-4-5-20251001": {
        "input_tokens": 90,
        "output_tokens": 35,
        "total_tokens": 125,
    },
}
```

The exact fields depend on the `UsageMetadata` returned by the model integration.

## Methods

1. `__repr__`: Returns the string representation of the aggregated usage dictionary.
   - **Syntax:**
     ```python
     __repr__(
         self
     ) -> str
     ```
   - **Equivalent behavior:**
     ```python
     return str(
         self.usage_metadata
     )
     ```

2. `on_llm_end`: Collects usage metadata after an LLM or chat-model call finishes.
   * Examines only the first generation:
     ```python
     response.generations[0][0]
     ```
   * Continues only when that generation is a `ChatGeneration`.
   * Continues only when its message is an `AIMessage`.
   * Reads usage from:
     ```python
     message.usage_metadata
     ```
   * Reads the grouping key from:
     ```python
     message.response_metadata.get(
         "model_name"
     )
     ```
   * Ignores the result when usage metadata or the model name is absent.
   * Stores the first usage value directly.
   * Aggregates later usage for the same model using `add_usage`.
   * Protects dictionary mutations with `_lock`.
   - **Syntax:**
     ```python
     on_llm_end(
         self,
         response: LLMResult, # Completed model result
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

## Collection Flow

```text
Model call finishes
        |
        v
Read response.generations[0][0]
        |
        v
Is it a ChatGeneration?
        |
   No --+--> Ignore the result
        |
       Yes
        |
        v
Is generation.message an AIMessage?
        |
   No --+--> Ignore the result
        |
       Yes
        |
        v
Read message.usage_metadata
and response_metadata["model_name"]
        |
        v
Are both values present?
        |
   No --+--> Ignore the result
        |
       Yes
        |
        v
Acquire thread lock
        |
        v
New model name?
   |             |
  Yes           No
   |             |
Store usage   Combine with add_usage
```

## Aggregation Behavior

### First Call for a Model

```python
self.usage_metadata[
    model_name
] = usage_metadata
```

### Later Call for the Same Model

```python
self.usage_metadata[
    model_name
] = add_usage(
    self.usage_metadata[
        model_name
    ],
    usage_metadata,
)
```

`add_usage` recursively adds standard token totals and compatible nested token-detail values.

Example:

```python
first = {
    "input_tokens": 10,
    "output_tokens": 5,
    "total_tokens": 15,
}

second = {
    "input_tokens": 7,
    "output_tokens": 3,
    "total_tokens": 10,
}
```

Aggregated result:

```python
{
    "input_tokens": 17,
    "output_tokens": 8,
    "total_tokens": 25,
}
```

## Thread Safety

Updates to `usage_metadata` are performed inside:

```python
with self._lock:
    ...
```

This prevents simultaneous completed model calls from mutating the dictionary at the same time.

Direct reads of:

```python
callback.usage_metadata
```

and calls to:

```python
repr(callback)
```

do not explicitly acquire the lock.

## What Is Ignored

The callback does not record usage when:

* `response.generations` is empty.
* The first generation list is empty.
* The first generation is not a `ChatGeneration`.
* The generation message is not an `AIMessage`.
* `AIMessage.usage_metadata` is missing or `None`.
* `AIMessage.response_metadata["model_name"]` is missing or falsey.

It also does not inspect later generations such as:

```python
response.generations[0][1]
response.generations[1][0]
```

Only:

```python
response.generations[0][0]
```

is considered.

## Error Handling

The method catches:

* `IndexError` while reading the first generation.
* `AttributeError` while reading the chat generation or message fields.

Other unexpected exceptions are not explicitly caught by this method.

## Direct Usage Example

```python
from langchain.chat_models import (
    init_chat_model,
)
from langchain_core.callbacks import (
    UsageMetadataCallbackHandler,
)

llm_1 = init_chat_model(
    model="openai:gpt-5.5"
)

llm_2 = init_chat_model(
    model=(
        "anthropic:"
        "claude-haiku-4-5-20251001"
    )
)

callback = (
    UsageMetadataCallbackHandler()
)

llm_1.invoke(
    "Hello",
    config={
        "callbacks": [
            callback
        ]
    },
)

llm_2.invoke(
    "Hello",
    config={
        "callbacks": [
            callback
        ]
    },
)

print(
    callback.usage_metadata
)
```

# `get_usage_metadata_callback`

Context manager that creates a `UsageMetadataCallbackHandler` and automatically registers it through LangChain's callback-configuration context.

```python
get_usage_metadata_callback(
    name: str = (
        "usage_metadata_callback"
    )
) -> Generator[
    UsageMetadataCallbackHandler,
    None,
    None
]
```

- Decorator: `@contextmanager`
- Added in: `langchain-core` `0.3.49`

## Parameters

* `name` — Name assigned to the internal `ContextVar`.
  * Default:
    ```text
    usage_metadata_callback
    ```

## Yields

A newly created:

```python
UsageMetadataCallbackHandler
```

The yielded callback can be inspected during or after the `with` block.

## Behavior

The context manager:

1. Creates a context variable:
   ```python
   ContextVar(
       name,
       default=None
   )
   ```
2. Registers it as an inheritable callback-configuration hook:
   ```python
   register_configure_hook(
       usage_metadata_callback_var,
       inheritable=True,
   )
   ```
3. Creates a new usage callback.
4. Stores that callback in the context variable.
5. Yields the callback.
6. Sets the context variable to `None` after normal completion.

## Context Flow

```text
Enter with block
        |
        v
Create ContextVar
        |
        v
Register inheritable configure hook
        |
        v
Create UsageMetadataCallbackHandler
        |
        v
Store callback in ContextVar
        |
        v
Model calls configured in this context
inherit the callback
        |
        v
Inspect cb.usage_metadata
        |
        v
Exit normally and set ContextVar to None
```

## Context-Manager Example

```python
from langchain.chat_models import (
    init_chat_model,
)
from langchain_core.callbacks import (
    get_usage_metadata_callback,
)

llm_1 = init_chat_model(
    model="openai:gpt-5.5"
)

llm_2 = init_chat_model(
    model=(
        "anthropic:"
        "claude-haiku-4-5-20251001"
    )
)

with get_usage_metadata_callback() as cb:
    llm_1.invoke(
        "Hello"
    )

    llm_2.invoke(
        "Hello"
    )

    print(
        cb.usage_metadata
    )
```

Unlike direct registration, the model calls do not need:

```python
config={
    "callbacks": [cb]
}
```

because the callback is made available through the callback-configuration context.

## Custom Context Variable Name

```python
with get_usage_metadata_callback(
    name="request_usage"
) as callback:
    model.invoke(
        "Explain callbacks."
    )
```

The name identifies the internally created context variable. It does not change the keys in `callback.usage_metadata`.

## Inheritance

The callback hook is registered using:

```python
inheritable=True
```

This allows callback configuration created within the context to propagate the handler to nested child runs.

## Exceptional Exit Detail

At this pinned source revision, cleanup is written as:

```python
usage_metadata_callback_var.set(
    cb
)

yield cb

usage_metadata_callback_var.set(
    None
)
```

It is not wrapped in an explicit `try`/`finally` block.

Therefore, when an exception is raised inside the `with` block and propagates through the context manager, the final:

```python
set(None)
```

statement may not execute for that context variable.

## Direct Callback vs Context Manager

| Approach | Callback attachment |
|---|---|
| `UsageMetadataCallbackHandler()` | Pass explicitly through `config={"callbacks": [...]}` |
| `get_usage_metadata_callback()` | Automatically available through the active callback configuration context |

Both approaches produce the same callback type and the same `usage_metadata` aggregation behavior.

## Important Limitations

* Usage is grouped only by `response_metadata["model_name"]`.
* Calls without that metadata key are ignored.
* Only the first `ChatGeneration` is inspected.
* Legacy token usage stored only in `LLMResult.llm_output` is not collected.
* Plain non-chat `Generation` objects are ignored.
* The callback tracks tokens, not monetary cost.
* The callback does not reset its dictionary automatically during its lifetime.
* Different model-name strings create separate dictionary entries, even when they refer to similar model variants.

## Module API

This module defines:

```python
UsageMetadataCallbackHandler
get_usage_metadata_callback
```

It does not define an explicit module-level `__all__` list.

## Source

This reference follows the pinned LangChain source:

```text
libs/core/langchain_core/callbacks/usage.py
Commit: 1c3a4186cf2ba4f28face59118ac7786de009f91
```